In [1]:
#滞销90

import pandas as pd
import numpy as np

# 读取原始 Excel 文件
df = pd.read_excel('C:\\Users\\Administrator\\Desktop\\销售订单4.5.xlsx')
data = pd.read_excel("C:\\Users\\Administrator\\Desktop\\shopee-listing4.5.xlsx")
dy = pd.read_excel("C:\\Users\\Administrator\\Desktop\\当前库存4.5.xlsx")
start_date_1 = pd.to_datetime('2026-03-30')
end_date_1 = pd.to_datetime('2026-04-05')


dy1 = dy[dy['仓库'].isin(['10.3 印尼海外总仓'])]
dy2 = dy[dy['仓库'].isin(['57 佛山海外仓(一仓)'])]
dy5 = dy[dy['仓库'].isin(['10.3 印尼海外总仓'])]

#不包括的SKU
dy1 = dy1[~dy1['SKU'].isin(['FMA0001-SD','HQA0001-QP','HQE0001-QP','HQE0001-SD'])]

dy3 = dy1[['SKU','商品名称','开发归属人','商品成本','可用数量','库龄']]

dy3 = dy3.rename(columns={'可用数量': '库存数量'})  
dy3['库存金额'] = dy3['库存数量'] * dy3['商品成本']

dx = dy3[['SKU','商品名称','开发归属人', '商品成本','库存数量','库存金额','库龄']]
dx[['商品成本', '库存金额']] = dx[['商品成本', '库存金额']].applymap(lambda x: int(x))


dy4 = dy1[['SKU','调拨未入库']]
dy4 = dy4.merge(dy2[['SKU','库存数量','采购未入库']], on='SKU', how='left')
dy4['在途数量'] = dy4['库存数量'] + dy4['调拨未入库'] + dy4['采购未入库']
dy4 = dy4.drop(['库存数量', '调拨未入库', '采购未入库'], axis=1)
dx = dx.merge(dy4, on='SKU', how='left')

dx = dx[dx['库存数量'] > 0]
dx = dx[dx['库龄'] > 90]


#在销售订单中加入店铺归属人，并清洗
    

df = df[~df['财务状态'].isin(['已退款'])]
df = df[~df['中文报关名'].isin(['冲榜虚拟商品'])]
df = df[~df['SKU'].isin(['FMA0001-SD','HQA0001-QP','HQE0001-QP','HQE0001-SD'])]

df['业务日期(年月日)'] = pd.to_datetime(df['业务日期(年月日)'])

#筛选日期
df = df[(df['业务日期(年月日)'] >= start_date_1) & (df['业务日期(年月日)'] <= end_date_1)]

# 去重 SKU 列,获取唯一 SKU 值
unique_sku = dx['SKU'].unique()

# 获取所有店铺名称,作为新表格的索引
stores = df['店铺归属人'].unique()


# 创建新的 DataFrame,初始化为 0
new_df = pd.DataFrame(columns=stores, index=unique_sku, data=0)
df['销售金额'] = df['销售金额']*0.000453

# 遍历原始数据,累加销售数量
for index, row in df.iterrows():
    sku = row['SKU']
    store = row['店铺归属人']
    sales_qty = row['销售金额']
    if sku in new_df.index:
        new_df.loc[sku, store] += sales_qty


new_df_reset = new_df.reset_index().rename(columns={'index': 'SKU'})
dx = dx.merge(new_df_reset, on='SKU', how='left')
#dx = dx.merge(new_df, on='SKU', how='left')

dx1 = dx[dx['开发归属人'].isin(['彭朋'])]
dx1= dx1.drop(columns=['喻天豪', '林逸铭', '彭子晨','梁辰雨','陆婧'])
dx2 = dx[dx['开发归属人'].isin(['余浩文','肖熙-虚拟账号'])]
dx2= dx2.drop(columns=['刘雨倩', '朱曼姣', '任强','袁豪','陆婧'])


# 保存新的 Excel 文件
with pd.ExcelWriter('C:\\Users\\Administrator\\Desktop\\滞销90.xlsx', engine='openpyxl') as writer:
    dx1.to_excel(writer, sheet_name='一组', index=False)
    dx2.to_excel(writer, sheet_name='二组', index=False)


D:\anaconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\1975214885.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  dx[['商品成本', '库存金额']] = dx[['商品成本', '库存金额']].applymap(lambda x: int(x))
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\1975214885.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '11.180946' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  new_df.loc[sku, store] += sales_qty
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\1975214885.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '14.949' has d

In [2]:
#新品情况

import pandas as pd
import numpy as np


dy = pd.read_excel("C:\\Users\\Administrator\\Desktop\\当前库存4.5.xlsx")

dy = dy[~dy['SKU'].isin(['FMA0001-SD','HQA0001-QP','HQE0001-SD'])]

dy1 = dy[dy['仓库'].isin(['10.3 印尼海外总仓'])]
dy2 = dy[dy['仓库'].isin(['57 佛山海外仓(一仓)'])]

dy1 = dy1[(dy1['库龄'] <= 30) & (dy1['库龄'] > 0)]
dy1 = dy1[dy1['30天销量'] == dy1['365天销量']]
dy1 = dy1[dy1['库存数量'] > 0]

dy1 = dy1[['SKU','商品名称','商品成本','库龄','5天日均销量','30天销量','365天销量','可用数量','调拨未入库','开发组']]

dy1 = dy1.merge(dy2[['SKU','库存数量','采购未入库']], on='SKU', how='left')

dy1['在途数量'] = dy1['库存数量'] + dy1['调拨未入库'] + dy1['采购未入库']

dy1 = dy1.drop(['库存数量', '调拨未入库', '采购未入库'], axis=1)

#dy1 = dy1[['SKU','商品名称','库龄','库存数量','5天日均销量','30天销量','365天销量','库存数量','调拨未入库']]
#dy1.to_excel("C:\\Users\\Administrator\\Desktop\\新品情况5.29.xlsx", index=False)

dy_zhuang = dy1[(dy1["开发组"] == "牧智康ZIPKINMU-印尼一组-彭朋") | (dy1["开发组"] == "牧智康ZIPKINMU-印尼一组-陈柳静")]
dy_yu = dy1[(dy1["开发组"] == "牧智康ZIPKINMU-印尼二组-余浩文") | (dy1["开发组"] == "牧智康ZIPKINMU-印尼一组-肖熙")]

dy_zhuang = dy_zhuang.drop(['开发组'], axis=1)
dy_yu = dy_yu.drop(['开发组'], axis=1)

with pd.ExcelWriter('C:\\Users\\Administrator\\Desktop\\新品情况3.xlsx', engine='openpyxl') as writer:
    dy_zhuang.to_excel(writer, sheet_name='一组', index=False)
    dy_yu.to_excel(writer, sheet_name='二组', index=False)



In [3]:
#运营周报

#店铺归属人周报销售-终极版

import pandas as pd
import numpy as np

df = pd.read_excel("C:\\Users\\Administrator\\Desktop\\销售订单4.5.xlsx")
data = pd.read_excel("C:\\Users\\Administrator\\Desktop\\shopee-listing4.5.xlsx")
dy = pd.read_excel("C:\\Users\\Administrator\\Desktop\\当前库存4.5.xlsx")


dx_a1 = pd.read_excel('C:\\Users\\Administrator\\Desktop\\滞销90.xlsx', sheet_name='一组')  
dx_a2 = pd.read_excel('C:\\Users\\Administrator\\Desktop\\滞销90.xlsx', sheet_name='二组') 

#在销售订单中加入店铺归属人，并清洗


df = df[~df['财务状态'].isin(['已退款'])]
df = df[~df['平台状态'].isin(['交易关闭'])]
df = df[~df['中文报关名'].isin(['冲榜虚拟商品'])]

#筛选日期

df['业务日期(年月日)'] = pd.to_datetime(df['业务日期(年月日)'])

start_date_1 = pd.to_datetime('2026-03-30')
end_date_1 = pd.to_datetime('2026-04-05')
df_1 = df[(df['业务日期(年月日)'] >= start_date_1) & (df['业务日期(年月日)'] <= end_date_1)]

start_date_2 = pd.to_datetime('2026-03-23')
end_date_2 = pd.to_datetime('2026-03-29')
df_2 = df[(df['业务日期(年月日)'] >= start_date_2) & (df['业务日期(年月日)'] <= end_date_2)]

start_date_3 = pd.to_datetime('2026-03-07')
end_date_3 = pd.to_datetime('2026-04-05')
df_3 = df[(df['业务日期(年月日)'] >= start_date_3) & (df['业务日期(年月日)'] <= end_date_3)]

start_date_4 = pd.to_datetime('2026-02-05')
end_date_4 = pd.to_datetime('2026-03-06')
df_4 = df[(df['业务日期(年月日)'] >= start_date_4) & (df['业务日期(年月日)'] <= end_date_4)]

start_date_5 = pd.to_datetime('2026-04-01')
end_date_5 = pd.to_datetime('2026-04-05')
df_5 = df[(df['业务日期(年月日)'] >= start_date_5) & (df['业务日期(年月日)'] <= end_date_5)]



#生成新的excel
#df = df.rename(columns={'业绩归属人': '店铺归属人'})
unique_stores = df['店铺归属人'].unique()
new_df = pd.DataFrame({'店铺归属人': ['刘雨倩','朱曼姣','袁豪','陆婧','任强','林逸铭','梁辰雨','彭子晨','喻天豪']})

#计算合计值
sales_sum_1 = df_1.groupby('店铺归属人')['销售金额'].sum().reset_index()
sales_sum_1 = sales_sum_1.rename(columns={'销售金额': '近7天销售金额合计'})

sales_sum_2 = df_2.groupby('店铺归属人')['销售金额'].sum().reset_index()
sales_sum_2 = sales_sum_2.rename(columns={'销售金额': '上7天销售金额合计'})

sales_sum_3 = df_3.groupby('店铺归属人')['销售金额'].sum().reset_index()
sales_sum_3 = sales_sum_3.rename(columns={'销售金额': '近30天销售金额合计'})

sales_sum_4 = df_4.groupby('店铺归属人')['销售金额'].sum().reset_index()
sales_sum_4 = sales_sum_4.rename(columns={'销售金额': '上30天销售金额合计'})

sales_sum_5 = df_5.groupby('店铺归属人')['销售金额'].sum().reset_index()
sales_sum_5 = sales_sum_5.rename(columns={'销售金额': '本月销售金额合计'})

new_df = new_df.merge(sales_sum_1, on='店铺归属人', how='left')
new_df = new_df.merge(sales_sum_2, on='店铺归属人', how='left')
new_df = new_df.merge(sales_sum_3, on='店铺归属人', how='left')
new_df = new_df.merge(sales_sum_4, on='店铺归属人', how='left')
new_df = new_df.merge(sales_sum_5, on='店铺归属人', how='left')

new_df = new_df.fillna(0)
for col in new_df.select_dtypes(include=['int64', 'float64']).columns:
    new_df[col] = new_df[col] * 0.00046

#本周成本计算
sales_sum_6 = df_1.groupby('店铺归属人')['成本'].sum().reset_index()
sales_sum_6 = sales_sum_6.rename(columns={'成本': '本周成本费用'})
new_df = new_df.merge(sales_sum_6, on='店铺归属人', how='left')


#当月动销数的计算
df_3['母SKU'] = df_3['SKU'].str.split('-').str[0]

store_sales = {}
for store in unique_stores:
    store_df = df_3[df_3['店铺归属人'] == store]
    store_sku_sales = store_df.groupby('母SKU')['销售数量'].count()
    dynamic_skus = store_sku_sales[store_sku_sales >= 10].index
    dynamic_count = len(dynamic_skus)
    store_sales[store] = dynamic_count


sales_sum_7 = pd.DataFrame.from_dict(store_sales, orient='index').reset_index()
sales_sum_7.columns = ['店铺归属人', '动销产品数']
new_df = new_df.merge(sales_sum_7, on='店铺归属人', how='left')

#店铺有的母SKU数

filled_column = '商品编码'
to_fill_column = '店铺SKU'

data[to_fill_column].fillna(data[filled_column], inplace=True)
data['SKU'] = data['店铺SKU'].str.split('-').str[0]

store_unique_sku = data.groupby('店铺归属人')['SKU'].nunique().to_dict()
store_data = pd.DataFrame.from_dict(store_unique_sku, orient='index').reset_index()
store_data.columns = ['店铺归属人', '母SKU数']
new_df = new_df.merge(store_data, on='店铺归属人', how='left')

#滞销率计算

new_df['动销率'] = new_df['动销产品数'] / new_df['母SKU数']
new_df['动销率'] = new_df['动销率'] * 100  # 将常数值转换为百分比
new_df['动销率'] = new_df['动销率'] .map(lambda x: '{:.2f}%'.format(x)) 

#库存金额计算（增加周转列和绘制运营-sku表）

dy['周转'] = 2*dy['库存数量'] / (dy['5天日均销量']+dy['15天日均销量'])
dy1 = dy[dy['仓库'].isin(['10.3 印尼海外总仓'])]
dy2 = dy[dy['仓库'].isin(['57 佛山海外仓(一仓)'])]


row_values = data['店铺SKU'].unique()
column_values = data['店铺归属人'].unique()

mat = pd.DataFrame(0, columns=column_values, index=row_values)

for _, row in data.iterrows():
    mat.loc[row['店铺SKU'], row['店铺归属人']] = 1

mat.reset_index(inplace=True)
mat.rename(columns={'index': 'SKU'}, inplace=True)
mat['加总'] = mat.iloc[:, 1:].sum(axis=1)

#在库金额的计算

matrix = mat.merge(dy1[['SKU', '库存金额']], left_on='SKU', right_on='SKU', how='left')
matrix['库存金额'].fillna(0, inplace=True)

matrix['库存金额除以加总'] = matrix['库存金额'] / matrix['加总']
for i in range(len(matrix)):
    matrix.iloc[i, :-3] = matrix.iloc[i, :-3].replace(1, matrix.loc[i, '库存金额除以加总'])
matrix = matrix.iloc[:, :-3]
matrix= matrix.T
matrix.columns = matrix.iloc[0]
matrix = matrix[1:]
matrix.reset_index(inplace=True)
matrix.rename(columns={'index': '店铺归属人'}, inplace=True)

matrix['在库金额'] = matrix.iloc[:, 1:].sum(axis=1)
matrix = matrix[['店铺归属人', '在库金额']]
new_df = new_df.merge(matrix, on='店铺归属人', how='left')


#在途金额的计算

matrix1 = mat.merge(dy2[['SKU', '在途金额']], left_on='SKU', right_on='SKU', how='left')
matrix1 = matrix1.merge(dy2[['SKU', '采购未入库金额']], left_on='SKU', right_on='SKU', how='left')
matrix1 = matrix1.merge(dy1[['SKU', '调拨未入库金额']], left_on='SKU', right_on='SKU', how='left')
matrix1.fillna(0, inplace=True)
matrix1['总在途'] = matrix1.iloc[:, -3:].sum(axis=1)
matrix1.drop(matrix1.columns[-4:-1], axis=1, inplace=True)

matrix1['在途金额除以加总'] = matrix1['总在途'] / matrix1['加总']
for i in range(len(matrix1)):
    matrix1.iloc[i, :-3] = matrix1.iloc[i, :-3].replace(1, matrix1.loc[i, '在途金额除以加总'])
matrix1 = matrix1.iloc[:, :-3]
matrix1= matrix1.T
matrix1.columns = matrix1.iloc[0]
matrix1 = matrix1[1:]
matrix1.reset_index(inplace=True)
matrix1.rename(columns={'index': '店铺归属人'}, inplace=True)

matrix1['在途金额'] = matrix1.iloc[:, 1:].sum(axis=1)
matrix1 = matrix1[['店铺归属人', '在途金额']]
new_df = new_df.merge(matrix1, on='店铺归属人', how='left')

#总库存金额
new_df['总库存金额'] = new_df['在库金额'] + new_df['在途金额']




#滞销期

total1 = dx_a1[dx_a1['库龄'] >= 120]['库存金额'].sum()
value1 = total1 / 4

total2 = dx_a2[dx_a2['库龄'] >= 120]['库存金额'].sum()
value2 = total2 / 4

new_df['滞销金额'] = 0


front_4_indices = [0, 1, 2, 4]  # 刘雨倩、朱曼姣、郑星怡、任强
for i in front_4_indices:
    new_df.loc[i, '滞销金额'] = value1

back_4_indices = [5, 6, 7, 8]  # 林逸铭、梁辰雨、彭子晨、喻天豪
for i in back_4_indices:
    new_df.loc[i, '滞销金额'] = value2


dx_a1_filtered = dx_a1[dx_a1['库龄'] >= 120]
dx_a2_filtered = dx_a2[dx_a2['库龄'] >= 120]


new_df['清仓金额'] = 0

people_a1 = ['刘雨倩', '朱曼姣', '袁豪', '任强']
for i, person in enumerate(people_a1):
    if person in dx_a1_filtered.columns:
        total = dx_a1_filtered[person].sum()
        # 找到在new_df中的对应位置
        idx = new_df[new_df['店铺归属人'] == person].index[0]
        new_df.loc[idx, '清仓金额'] = total
        print(f"{person}的清仓金额: {total:.2f}")
    else:
        print(f"警告: dx_a1中找不到列 '{person}'")

people_a2 = ['林逸铭', '梁辰雨', '彭子晨', '喻天豪']
for i, person in enumerate(people_a2):
    if person in dx_a2_filtered.columns:
        total = dx_a2_filtered[person].sum()
        idx = new_df[new_df['店铺归属人'] == person].index[0]
        new_df.loc[idx, '清仓金额'] = total
        print(f"{person}的清仓金额: {total:.2f}")
    else:
        print(f"警告: dx_a2中找不到列 '{person}'")


#订单数

sales_sum_8 = df_1.groupby('店铺归属人')['平台单号'].nunique().reset_index()
new_df = new_df.merge(sales_sum_8, on='店铺归属人', how='left')
new_df = new_df.rename(columns={'平台单号': '订单数'})
              

# 保存修改后的数据框架到新的Excel文件

# 打印矩阵
new_df.to_excel("C:\\Users\\Administrator\\Desktop\\店铺归属人周报.xlsx", index=False)


D:\anaconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\3917418731.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3['母SKU'] = df_3['SKU'].str.split('-').str[0]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\3917418731.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

Fo

刘雨倩的清仓金额: 22196.84
朱曼姣的清仓金额: 37159.38
袁豪的清仓金额: 20267.39
任强的清仓金额: 15192.91
林逸铭的清仓金额: 12026.58
梁辰雨的清仓金额: 4947.25
彭子晨的清仓金额: 21350.49
喻天豪的清仓金额: 2840.67


C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\3917418731.py:202: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '44013.25' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  new_df.loc[i, '滞销金额'] = value1
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\3917418731.py:221: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '22196.84144999996' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  new_df.loc[idx, '清仓金额'] = total


In [4]:
#开发周报销售

import pandas as pd
df = pd.read_excel("C:\\Users\\Administrator\\Desktop\\销售订单4.5.xlsx")
dy = pd.read_excel("C:\\Users\\Administrator\\Desktop\\当前库存4.5.xlsx")
data = pd.read_excel("C:\\Users\\Administrator\\Desktop\\商品信息4.5.xlsx")

start_time = pd.to_datetime('2026-03-30').date()
end_time = pd.to_datetime('2026-04-05').date()

df = df[~df['财务状态'].isin(['已退款'])]
df = df[~df['中文报关名'].isin(['冲榜虚拟商品'])]
df = df[~df['平台状态'].isin(['交易关闭'])]


df['开发'] = df['SKU'].str[:3]
dy['开发'] = dy['SKU'].str[:3]
data['开发'] = data['SKU'].str[:3]

dy['母SKU'] = dy['SKU'].str.split('-').str[0]
dy1 = dy[dy['仓库'].isin(['10.3 印尼海外总仓'])]
dy2 = dy[dy['仓库'].isin(['57 佛山海外仓(一仓)'])]
dy1 = dy1[~dy1['SKU'].isin(['FMA0001-SD','HQA0001-QP','HQE0001-QP'])]
dy2 = dy2[~dy2['SKU'].isin(['FMA0001-SD','HQA0001-QP','HQE0001-QP'])]


#筛选日期

df['业务日期(年月日)'] = pd.to_datetime(df['业务日期(年月日)'])

start_date_1 = pd.to_datetime('2026-03-30')
end_date_1 = pd.to_datetime('2026-04-05')
df_1 = df[(df['业务日期(年月日)'] >= start_date_1) & (df['业务日期(年月日)'] <= end_date_1)]

start_date_2 = pd.to_datetime('2026-03-23')
end_date_2 = pd.to_datetime('2026-03-29')
df_2 = df[(df['业务日期(年月日)'] >= start_date_2) & (df['业务日期(年月日)'] <= end_date_2)]

start_date_3 = pd.to_datetime('2026-03-07')
end_date_3 = pd.to_datetime('2026-04-05')
df_3 = df[(df['业务日期(年月日)'] >= start_date_3) & (df['业务日期(年月日)'] <= end_date_3)]

start_date_4 = pd.to_datetime('2026-02-05')
end_date_4 = pd.to_datetime('2026-03-06')
df_4 = df[(df['业务日期(年月日)'] >= start_date_4) & (df['业务日期(年月日)'] <= end_date_4)]

start_date_5 = pd.to_datetime('2026-04-01')
end_date_5 = pd.to_datetime('2026-04-05')
df_5 = df[(df['业务日期(年月日)'] >= start_date_5) & (df['业务日期(年月日)'] <= end_date_5)]


#生成新的excel


new_df = pd.DataFrame({'开发': ['HQC','HQ0','HQB','FMA','HQD','HQA','FMB','HQE','HQH','HQI']})


#计算合计值
sales_sum_1 = df_1.groupby('开发')['销售金额'].sum().reset_index()
sales_sum_1 = sales_sum_1.rename(columns={'销售金额': '本周销售金额合计'})

sales_sum_2 = df_2.groupby('开发')['销售金额'].sum().reset_index()
sales_sum_2 = sales_sum_2.rename(columns={'销售金额': '上周销售金额合计'})

sales_sum_3 = df_3.groupby('开发')['销售金额'].sum().reset_index()
sales_sum_3 = sales_sum_3.rename(columns={'销售金额': '本月销售金额合计'})

sales_sum_4 = df_4.groupby('开发')['销售金额'].sum().reset_index()
sales_sum_4 = sales_sum_4.rename(columns={'销售金额': '上月销售金额合计'})

sales_sum_5 = df_5.groupby('开发')['销售金额'].sum().reset_index()
sales_sum_5 = sales_sum_5.rename(columns={'销售金额': '8月销售金额合计'})


df_3.loc[:, '母SKU'] = df_3['SKU'].str.split('-').str[0]
sku_count = df_3.groupby(['开发','母SKU'])['母SKU'].count().reset_index(name='SKU_count')
over_10_sku = sku_count[sku_count['SKU_count'] > 10].groupby('开发')['母SKU'].count()
sales_sum_6 = over_10_sku.rename('动销数')


#导入新excel
new_df = new_df.merge(sales_sum_1, on='开发', how='left')
new_df = new_df.merge(sales_sum_2, on='开发', how='left')
new_df = new_df.merge(sales_sum_3, on='开发', how='left')
new_df = new_df.merge(sales_sum_4, on='开发', how='left')
new_df = new_df.merge(sales_sum_5, on='开发', how='left')

#空白处填写0，将所有元素乘上0.00046
new_df = new_df.fillna(0)
for col in new_df.select_dtypes(include=['int64', 'float64']).columns:
    new_df[col] = new_df[col] * 0.00046

new_df = new_df.merge(sales_sum_6, on='开发', how='left')

# 店铺名称作为行名称
new_df.set_index('开发', inplace=True)
new_df = new_df.reset_index()

#金额

#海外在库金额
sales_sum_1 = dy1.groupby('开发')['库存金额'].sum().reset_index()
sales_sum_1 = sales_sum_1.rename(columns={'库存金额': '在库金额'})
new_df = new_df.merge(sales_sum_1, on='开发', how='left')

#采购金额
sales_sum_2 = dy2.groupby('开发')['在途金额'].sum().reset_index()
sales_sum_2 = sales_sum_2.rename(columns={'在途金额': '采购金额'})
new_df = new_df.merge(sales_sum_2, on='开发', how='left')

#国内在库
sales_sum_3 = dy2.groupby('开发')['库存金额'].sum().reset_index()
sales_sum_3 = sales_sum_3.rename(columns={'库存金额': '国内在库金额'})
new_df = new_df.merge(sales_sum_3, on='开发', how='left')

#调拨金额
sales_sum_4 = dy1.groupby('开发')['在途金额'].sum().reset_index()
sales_sum_4 = sales_sum_4.rename(columns={'在途金额': '调拨金额'})
new_df = new_df.merge(sales_sum_4, on='开发', how='left')

#总在途金额
new_df['总在途金额'] = new_df['采购金额'] + new_df['国内在库金额'] + new_df['调拨金额']


#数量

#海外在库数量
sales_sum_5 = dy1.groupby('开发')['库存数量'].sum().reset_index()
sales_sum_5 = sales_sum_5.rename(columns={'库存数量': '在库数量'})
new_df = new_df.merge(sales_sum_5, on='开发', how='left')

#采购未入库
sales_sum_6 = dy2.groupby('开发')['采购未入库'].sum().reset_index()
sales_sum_6 = sales_sum_6.rename(columns={'采购未入库': '采购数量'})
new_df = new_df.merge(sales_sum_6, on='开发', how='left')

#国内在库数量
sales_sum_7 = dy2.groupby('开发')['库存数量'].sum().reset_index()
sales_sum_7 = sales_sum_7.rename(columns={'库存数量': '国内库存数量'})
new_df = new_df.merge(sales_sum_7, on='开发', how='left')

#调拨数量
sales_sum_8 = dy1.groupby('开发')['调拨未入库'].sum().reset_index()
sales_sum_8 = sales_sum_8.rename(columns={'调拨未入库': '调拨数量'})
new_df = new_df.merge(sales_sum_8, on='开发', how='left')

#总在途数量
new_df['总在途数量'] = new_df['采购数量'] + new_df['国内库存数量'] +new_df['调拨数量']

#销量

#平均销量
sales_sum_5 = dy1.groupby('开发')['日平均销量'].sum().reset_index()
sales_sum_5 = sales_sum_5.rename(columns={'日平均销量': '平均销量'})
new_df = new_df.merge(sales_sum_5, on='开发', how='left')

#sku数量

dy11 = dy1.groupby('母SKU')['库存数量'].sum().reset_index()
dy11['开发'] = dy11['母SKU'].str[:3]
sales_sum_9 = dy11.loc[dy11["库存数量"] > 0].groupby("开发")["库存数量"].count()
sales_sum_9 = sales_sum_9.rename('在库sku数量')
new_df = new_df.merge(sales_sum_9, on='开发', how='left')

#母sku

#母SKU数
unique_sku = data.groupby('开发')['母SKU'].nunique().to_dict()
store_data = pd.DataFrame.from_dict(unique_sku, orient='index').reset_index()
store_data.columns = ['开发', '母SKU数']
new_df = new_df.merge(store_data, on='开发', how='left')


#本周审核的母SKU数
data['创建时间'] = pd.to_datetime(data['创建时间'], format='%Y-%m-%d %H:%M:%S:%f')
data['日期'] = data['创建时间'].dt.date

data1 = data[(data['日期']>= start_time) & (data['日期']<= end_time)]

unique_sku1 = data1.groupby('开发')['母SKU'].nunique().to_dict()
store_data1 = pd.DataFrame.from_dict(unique_sku1, orient='index').reset_index()
store_data1.columns = ['开发', '本周开发数量']
new_df = new_df.merge(store_data1, on='开发', how='left')



#产品成本金额
sales_sum_10 = df_1.groupby('开发')['成本'].sum().reset_index()
sales_sum_10 = sales_sum_10.rename(columns={'成本': '产品成本金额'})
new_df = new_df.merge(sales_sum_10, on='开发', how='left')

new_df['在库周转'] = (new_df['在库金额'] / new_df['产品成本金额']) * 7
new_df['总周转'] = ((new_df['在库金额']+new_df['总在途金额']) / new_df['产品成本金额']) * 7

#new_df = new_df.drop(columns=['产品成本金额'])


#new_df = new_df.reset_index()

new_df.to_excel("C:\\Users\\Administrator\\Desktop\\开发周报.xlsx", index=False)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_13156\2676600210.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3.loc[:, '母SKU'] = df_3['SKU'].str.split('-').str[0]
